# NLQ with LangChain — Deep Dive

**Natural Language Query (NLQ)** lets a user ask a database a question in plain English and get a plain-English answer back — no SQL required.

This notebook teaches every concept from the ground up:

| Section | Concept |
|---------|---------|
| 0–2 | Install, API key, SQLite database setup |
| 3 | `SQLDatabase` — the Python wrapper LangChain uses to talk to any DB |
| 4 | **One-shot chain** — the simple approach: question → SQL → answer (one pass) |
| 5 | **SQL Agent** — the robust approach: iterative tool-calling loop |
| 6 | Chain vs Agent — side-by-side comparison |
| 7 | **Prompt engineering** — system prompts, business rules, few-shot examples |
| 8 | **Memory** — conversation history for follow-up questions |
| 9 | Agent internals — inspecting the prompt template and tools |
| 10 | **Mini chatbot** — putting everything together in ~40 lines |
| 11 | Exercises |
| 12 | Concept summary / cheat sheet |

**Self-contained:** Uses a SQLite file (`retail.db`) created inline. No PostgreSQL, no Docker.

**LLM:** [Groq](https://console.groq.com) (free tier, sign up takes 1 minute). Uses `llama-3.3-70b-versatile`.

## Section 0 — Install dependencies

Pinned to the exact versions this notebook was tested against.

| Package | Version | What it provides |
|---------|---------|-----------------|
| `langchain` | 1.2.18 | Core orchestration, LCEL pipe syntax |
| `langchain-community` | 0.4.1 | `SQLDatabase`, `create_sql_agent`, SQL prompt templates |
| `langchain-core` | 1.4.0 | `ChatPromptTemplate`, `StrOutputParser`, `RunnablePassthrough`, message history, `trim_messages` |
| `langchain-groq` | 1.1.2 | Groq LLM integration |
| `sqlalchemy` | 2.x | DB connection layer used internally by `SQLDatabase` |
| `pydantic` | 2.x | `SecretStr` — prevents API key from appearing in logs |
| `requests` | 2.32.4 | HTTP client — pinned to satisfy `google-colab 1.0.0` on Colab |

> **Note on `langchain.memory`:** This module was fully removed in LangChain 1.0.
> The old `ConversationBufferMemory`, `ConversationSummaryMemory` etc. no longer exist.
> Section 8 uses the current replacements: `InMemoryChatMessageHistory` and `trim_messages`.

In [ ]:
# Run this cell first. The -q flag suppresses the install noise.
# On Google Colab this will take ~60 seconds; on a local venv with packages already present it's instant.
#
# requests is pinned to 2.32.4 because google-colab 1.0.0 (pre-installed on Colab)
# declares an exact requirement on that version. Installing a newer requests causes
# a harmless resolver warning but can break Colab internal networking — pinning
# avoids both the warning and the risk. On local Jupyter this pin is equally safe.
%pip install -q \
    "langchain==1.2.18" \
    "langchain-community==0.4.1" \
    "langchain-core==1.4.0" \
    "langchain-groq==1.1.2" \
    "sqlalchemy>=2.0,<3.0" \
    "pydantic>=2.0,<3.0" \
    "requests==2.32.4"

In [ ]:
# Quick sanity check — prints installed vs expected version for each package.
# A ⚠ means you have a different version; the notebook may still work but wasn't tested against it.
import langchain, langchain_community, langchain_core, langchain_groq, sqlalchemy, pydantic

EXPECTED = {
    "langchain":           ("1.2.18", langchain.__version__),
    "langchain_community": ("0.4.1",  langchain_community.__version__),
    "langchain_core":      ("1.4.0",  langchain_core.__version__),
    "langchain_groq":      ("1.1.2",  langchain_groq.__version__),
    "sqlalchemy":          ("2.",     sqlalchemy.__version__),
    "pydantic":            ("2.",     pydantic.__version__),
}

print(f"{'Package':<22}  {'Expected':<10}  {'Installed':<12}  Status")
print("-" * 58)
for pkg, (exp, got) in EXPECTED.items():
    ok = got.startswith(exp)
    print(f"{pkg:<22}  {exp:<10}  {got:<12}  {'✓' if ok else '⚠ MISMATCH'}")

## Section 1 — API key configuration

Get a free Groq API key at <https://console.groq.com> (takes ~1 minute, no credit card).

**How to set the key:**

- **Google Colab:** Use *Runtime → Secrets* (the 🔑 icon in the left sidebar). Add a secret named `GROQ_API_KEY`. The cell below reads it automatically via `google.colab.userdata`.
- **Local Jupyter:** Run `export GROQ_API_KEY=gsk_...` in your shell before starting the notebook, or paste the key directly in the cell (but do not commit it).

The key is wrapped in `SecretStr` (from pydantic) which prevents it from appearing in logs or tracebacks.

In [ ]:
import os

# --- Colab: read from Secrets panel (Runtime → Secrets → add GROQ_API_KEY) ---
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    # Not on Colab — fall back to environment variable or inline value
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

# If neither worked, paste your key here (local use only — never commit this):
# GROQ_API_KEY = "gsk_..."

GROQ_MODEL = os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is not set. See the markdown cell above for instructions.")

# Mask key in output so it's safe to share a screenshot
masked = GROQ_API_KEY[:6] + "..." + GROQ_API_KEY[-4:]
print(f"API key loaded : {masked}")
print(f"Model          : {GROQ_MODEL}")

## Section 2 — Build the retail SQLite database

We create a self-contained SQLite file that mirrors the production PostgreSQL schema.

**Why SQLite?** Zero setup — no server, no Docker, works on Colab out of the box.
SQLite and PostgreSQL produce the same LangChain behavior; the only difference is date syntax
(`datetime('now', '-30 days')` instead of `NOW() - INTERVAL '30 days'`).

**Schema — 4 tables:**
```
customers   id, name, email, region, created_at
products    id, name, category, price
orders      id, customer_id→customers, status, ordered_at
order_items id, order_id→orders, product_id→products, quantity, unit_price
```

**Key design decision:** `unit_price` on `order_items` is a **snapshot** of the price at
the time of purchase — not a foreign key to `products.price`. This lets us compute accurate
historical revenue even after a product's price changes.

**Seeded test case:** Frank Lee has one *cancelled* order. Every correct sales/revenue query
must exclude it. This is the business rule we'll enforce via prompt engineering.

In [ ]:
import sqlite3
import pathlib

DB_PATH = pathlib.Path("retail.db")

# Delete any existing file so this cell is safe to re-run
DB_PATH.unlink(missing_ok=True)

conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

# ── Schema ────────────────────────────────────────────────────────────────────
cur.executescript("""
CREATE TABLE customers (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    name       TEXT    NOT NULL,
    email      TEXT    UNIQUE NOT NULL,
    region     TEXT    NOT NULL,           -- North | South | East | West
    created_at TEXT    DEFAULT (datetime('now'))
);

CREATE TABLE products (
    id       INTEGER PRIMARY KEY AUTOINCREMENT,
    name     TEXT    NOT NULL,
    category TEXT    NOT NULL,             -- Electronics | Office | Stationery
    price    REAL    NOT NULL
);

CREATE TABLE orders (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL REFERENCES customers(id),
    status      TEXT    NOT NULL,          -- pending | shipped | delivered | cancelled
    ordered_at  TEXT    NOT NULL
);

CREATE TABLE order_items (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id   INTEGER NOT NULL REFERENCES orders(id),
    product_id INTEGER NOT NULL REFERENCES products(id),
    quantity   INTEGER NOT NULL,
    unit_price REAL    NOT NULL  -- price snapshot at time of order, not products.price
);
""")

# ── Customers ─────────────────────────────────────────────────────────────────
cur.executemany("INSERT INTO customers (name, email, region) VALUES (?,?,?)", [
    ("Alice Martin", "alice@example.com", "North"),
    ("Bob Chen",     "bob@example.com",   "South"),
    ("Carol Davis",  "carol@example.com", "East"),
    ("Dan Okafor",   "dan@example.com",   "North"),
    ("Eva Singh",    "eva@example.com",   "West"),
    ("Frank Lee",    "frank@example.com", "South"),  # ← has a cancelled order
    ("Grace Kim",    "grace@example.com", "East"),
    ("Harry Brown",  "harry@example.com", "North"),  # ← no orders (inactive)
    ("Isla White",   "isla@example.com",  "West"),   # ← no orders (inactive)
])

# ── Products ──────────────────────────────────────────────────────────────────
cur.executemany("INSERT INTO products (name, category, price) VALUES (?,?,?)", [
    ("Wireless Mouse",      "Electronics", 29.99),
    ("Mechanical Keyboard", "Electronics", 89.99),
    ("USB-C Hub",           "Electronics", 49.99),
    ("Desk Lamp",           "Office",      34.99),
    ("Notebook (A5)",       "Stationery",   8.99),
    ("Ballpoint Pens x10",  "Stationery",   5.49),
    ("Laptop Stand",        "Electronics", 59.99),
    ("Webcam HD",           "Electronics", 79.99),
])

# ── Orders ────────────────────────────────────────────────────────────────────
# Orders 1-7 are within the last 30 days; orders 8-10 are 50-75 days ago
cur.executemany(
    "INSERT INTO orders (customer_id, status, ordered_at) VALUES (?,?,datetime('now',?))", [
    (1, "delivered", "-5 days"),
    (2, "shipped",   "-8 days"),
    (3, "pending",   "-2 days"),
    (4, "pending",   "-1 day"),
    (5, "delivered", "-15 days"),
    (6, "cancelled", "-20 days"),  # ← Frank's cancelled order — must be excluded from sales
    (7, "pending",   "-3 days"),
    (1, "delivered", "-60 days"),
    (2, "delivered", "-75 days"),
    (3, "delivered", "-50 days"),
])

# ── Order items ───────────────────────────────────────────────────────────────
cur.executemany(
    "INSERT INTO order_items (order_id, product_id, quantity, unit_price) VALUES (?,?,?,?)", [
    (1,  1, 2, 29.99),  # Alice:  2× Wireless Mouse
    (1,  3, 1, 49.99),  # Alice:  1× USB-C Hub
    (2,  2, 1, 89.99),  # Bob:    1× Mechanical Keyboard
    (2,  7, 1, 59.99),  # Bob:    1× Laptop Stand
    (3,  1, 1, 29.99),  # Carol:  1× Wireless Mouse
    (3,  4, 2, 34.99),  # Carol:  2× Desk Lamp
    (4,  8, 1, 79.99),  # Dan:    1× Webcam HD
    (5,  2, 2, 89.99),  # Eva:    2× Mechanical Keyboard
    (5,  5, 3,  8.99),  # Eva:    3× Notebook (A5)
    (6,  6, 5,  5.49),  # Frank:  5× Ballpoint Pens — CANCELLED, must not count in sales
    (7,  1, 3, 29.99),  # Grace:  3× Wireless Mouse
    (7,  3, 2, 49.99),  # Grace:  2× USB-C Hub
    (8,  2, 1, 89.99),  # Alice (old order)
    (9,  1, 1, 29.99),  # Bob   (old order)
    (10, 7, 1, 59.99),  # Carol (old order)
])

conn.commit()
conn.close()
print(f"retail.db created at {DB_PATH.resolve()}")

In [ ]:
# Verify the database was created correctly — sqlite3 was already imported above.

conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

print("Table row counts:")
for table in ["customers", "products", "orders", "order_items"]:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"  {table:<15} {cur.fetchone()[0]} rows")

# Confirm Frank's cancelled order exists — we'll use this as the test case later
cur.execute("SELECT c.name, o.status FROM orders o JOIN customers c ON o.customer_id = c.id WHERE o.status = 'cancelled'")
print("\nCancelled orders (should be Frank Lee's):")
for row in cur.fetchall():
    print(f"  {row}")

conn.close()

## Section 3 — `SQLDatabase`: the bridge between LangChain and your database

Before writing any LLM code, understand `SQLDatabase` — the object that sits between LangChain and the actual database.

**What it is:** A Python object (not an LLM). It:
1. Connects to any SQLAlchemy-compatible database (SQLite, PostgreSQL, MySQL, etc.)
2. Introspects the schema — table names, column names and types, sample rows
3. Executes SQL strings and returns results as **plain text**

**Why it matters:** The LLM never touches the database directly. The flow is always:
```
LLM generates SQL string
    → SQLDatabase.run(sql) executes it
        → returns plain text result
            → LLM reads the text and writes the answer
```

`db.dialect` is automatically injected into the agent's system prompt as `{dialect}`,
which tells the LLM to use SQLite-specific syntax (e.g. `datetime()`) rather than
PostgreSQL syntax (`NOW()`, `INTERVAL`).

In [ ]:
from langchain_community.utilities import SQLDatabase

# Connect — same call regardless of database type (swap the URI for PostgreSQL/MySQL)
db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")

print("── Dialect ──────────────────────────────────────────────────────")
# This value is injected into the agent's system prompt as {dialect}
# so the LLM knows to write SQLite-compatible SQL
print(db.dialect)

print("\n── Usable tables ────────────────────────────────────────────────")
print(db.get_usable_table_names())

print("\n── Schema for 'orders' and 'order_items' ────────────────────────")
# This is what the agent injects into the LLM context:
# CREATE TABLE statements + 3 sample rows per table
print(db.get_table_info(["orders", "order_items"]))

In [ ]:
# db.run() executes raw SQL and returns the result as a plain string.
# This is the exact text that gets sent back to the LLM.
result = db.run("SELECT name, region FROM customers LIMIT 4")
print("Result type :", type(result))
print("Result value:", result)
# Notice: it's a plain string, not a list of tuples.
# The LLM receives this string and parses it as natural language context.

**Key takeaway:** `SQLDatabase` is just a dumb executor. It runs whatever SQL string it receives and returns a string. All intelligence — deciding *what* SQL to write and *how* to interpret the results — comes from the LLM. The next two sections show two different ways to use it.

## Section 4 — One-shot chain (LCEL)

### The idea

The simplest possible NLQ pipeline has two steps:

```
question
  ──► [LLM call 1] generate SQL   (schema injected in system prompt)
  ──► db.run(sql)                  (execute against SQLite)
  ──► [LLM call 2] format answer   (turn raw rows into English)
  ──► answer
```

Total: **2 LLM calls**, single pass, no retries.

### What is LCEL?

LCEL (LangChain Expression Language) lets you compose steps with `|` (pipe):

```python
chain = step_a | step_b | step_c
chain.invoke({"key": "value"})
# input dict → step_a → step_b → step_c → final output
```

Each step can be a prompt template, an LLM, an output parser, or any Python callable
wrapped with `RunnableLambda`. `RunnablePassthrough.assign(key=fn)` adds a new key to
the dict flowing through the pipe without replacing existing keys.

### Trade-offs

| | One-shot chain |
|---|---|
| ✅ | Lowest latency (2 LLM calls) |
| ✅ | You see the exact SQL generated |
| ❌ | No error recovery — bad SQL = exception |
| ❌ | You must manually inject the schema into the prompt |
| ❌ | Struggles with complex multi-table joins |

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from pydantic import SecretStr

# ── LLM ───────────────────────────────────────────────────────────────────────
# temperature=0 → deterministic output. SQL must be exact, not creative.
llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,
    api_key=SecretStr(GROQ_API_KEY),
)

# ── Prompt 1: question + schema → SQL ─────────────────────────────────────────
# We inject the full schema into the system message so the LLM knows every
# table and column. The {schema} placeholder is filled by RunnablePassthrough.assign below.
sql_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a SQLite expert. Given the question and the database schema below, "
     "write a single syntactically correct SQLite query.\n\n"
     "Schema:\n{schema}\n\n"
     "Rules:\n"
     "- Output ONLY the raw SQL query. No markdown fences, no explanation.\n"
     "- Add LIMIT 10 unless the question specifies a different number.\n"
     "- Use datetime('now', '-N days') for relative dates (this is SQLite, not PostgreSQL).\n"),
    ("human", "{question}"),
])

# ── Prompt 2: question + SQL + results → English answer ───────────────────────
answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful data analyst. Given the original question, the SQL that was run, "
     "and the query results, write a clear plain-English answer. Be concise."),
    ("human",
     "Question : {question}\n\n"
     "SQL run  : {sql}\n\n"
     "Results  : {results}"),
])

# ── SQL execution helper ───────────────────────────────────────────────────────
def run_sql(sql: str) -> str:
    """Execute the generated SQL and return result as string. Catches errors gracefully."""
    # Strip accidental markdown fences the LLM sometimes adds despite instructions
    sql = sql.strip().strip("```sql").strip("```").strip()
    try:
        return db.run(sql)
    except Exception as exc:
        return f"SQL ERROR: {exc}"

# ── Stage 1: question → SQL ───────────────────────────────────────────────────
# RunnablePassthrough.assign adds a 'schema' key to the incoming dict
# by calling db.get_table_info() — the full schema is fetched once per question.
sql_chain = (
    RunnablePassthrough.assign(schema=lambda _: db.get_table_info())
    | sql_prompt
    | llm
    | StrOutputParser()   # extracts the text content from the LLM response
)

# ── Full pipeline: question → SQL → execute → answer ─────────────────────────
def run_chain(question: str) -> dict:
    """Run the one-shot chain and return sql, raw results, and final answer."""
    sql     = sql_chain.invoke({"question": question})
    results = run_sql(sql)
    answer  = (answer_prompt | llm | StrOutputParser()).invoke(
                  {"question": question, "sql": sql, "results": results})
    return {"sql": sql, "results": results, "answer": answer}

print("One-shot chain ready.")

In [ ]:
# Run a simple question — inspect every intermediate value
r = run_chain("How many customers are in each region?")

print("── SQL generated by LLM ─────────────────────────────────────────")
print(r["sql"])
print("\n── Raw result from db.run() ─────────────────────────────────────")
print(r["results"])
print("\n── Final English answer ─────────────────────────────────────────")
print(r["answer"])

In [ ]:
# The chain's blind spot: without explicit business rules in the prompt,
# the LLM may include Frank's CANCELLED order in sales totals.
# We'll fix this properly in Section 7 with a system prompt.
r2 = run_chain("What is the top selling product by quantity?")

print("── SQL (check: does it filter cancelled orders?) ────────────────")
print(r2["sql"])
print("\n── Answer ───────────────────────────────────────────────────────")
print(r2["answer"])

# Ground truth: Ballpoint Pens (Frank's cancelled order) must NOT appear
conn = sqlite3.connect(DB_PATH)
correct = conn.execute("""
    SELECT p.name, SUM(oi.quantity) AS total_qty
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.id
    JOIN products p ON oi.product_id = p.id
    WHERE o.status != 'cancelled'
    GROUP BY p.name
    ORDER BY total_qty DESC
    LIMIT 3
""").fetchall()
conn.close()

print("\n── Ground truth (cancelled excluded) ────────────────────────────")
for name, qty in correct:
    print(f"  {name}: {qty} units")

If the chain included Ballpoint Pens x10 in the result, the business rule was silently violated. The LLM had no instruction to exclude cancelled orders — it just did what seemed natural. Section 7 fixes this with an explicit system prompt.

## Section 5 — SQL Agent (`create_sql_agent`)

### The idea

Instead of one fixed pipeline, the agent lets the LLM **drive a loop** — it decides which tools to call, in what order, and when to stop.

```
question
  ──► LLM: "I need the table list"     → tool: sql_db_list_tables
  ──► LLM: "I need the schema"         → tool: sql_db_schema("orders, order_items, products")
  ──► LLM: "I'll run this SQL"         → tool: sql_db_query("SELECT ...")
         ↑                                       │
         └── if error: LLM sees error ◄──────────┘
                       and retries with corrected SQL
  ──► LLM: "I have enough — here's the answer"
```

### The four SQL tools the LLM can call

| Tool | What it does |
|------|-------------|
| `sql_db_list_tables` | Returns the list of all table names |
| `sql_db_schema` | Returns `CREATE TABLE` + 3 sample rows for given tables |
| `sql_db_query` | Executes a SQL string and returns the result |
| `sql_db_query_checker` | Asks the LLM to double-check SQL before running it |

### `agent_type` — why `"tool-calling"` matters

| `agent_type` | Mechanism | Reliability |
|---|---|---|
| `"tool-calling"` | LLM returns structured JSON function calls | Robust — native API support in all modern models |
| `"zero-shot-react-description"` | LLM outputs `Thought/Action/Observation` text, parsed by regex | Fragile — Llama 3 occasionally mis-formats this, causing crashes |

**Always use `"tool-calling"` with modern models.**

### Trade-offs

| | SQL Agent |
|---|---|
| ✅ | Self-correcting — retries on SQL errors |
| ✅ | Auto-discovers schema — no manual injection needed |
| ✅ | Handles complex multi-table joins reliably |
| ❌ | 3–5 LLM calls per question (~3–6s) |
| ❌ | Harder to inspect exactly what SQL ran (use `verbose=True`) |

In [ ]:
from langchain_community.agent_toolkits import create_sql_agent

# verbose=True prints every tool call the LLM makes — read this output carefully,
# it shows the full discover → schema → query → answer loop.
agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="tool-calling",   # structured JSON calls, not text-parsed ReAct
    top_k=20,                    # injects LIMIT 20 into every generated query
    verbose=True,
    agent_executor_kwargs={"handle_parsing_errors": True},
)

print("Agent ready.")

In [ ]:
# Same question as Section 4.
# Watch the verbose output above the answer — you'll see the agent calling
# sql_db_list_tables, then sql_db_schema, then sql_db_query in sequence.
result = agent.invoke({"input": "How many customers are in each region?"})

print("\n── Final answer ─────────────────────────────────────────────────")
print(result["output"])

In [ ]:
# Self-correction demo: ask for a column that doesn't exist by name.
# The agent will try, get a SQL error, inspect the schema, and retry.
# Watch the verbose output for the error → schema-check → retry loop.
result = agent.invoke({"input": "What is the average order_value per customer?"})

print("\n── Final answer ─────────────────────────────────────────────────")
print(result["output"])
# 'order_value' doesn't exist — the agent discovers it needs to compute
# SUM(quantity * unit_price) from order_items instead.

## Section 6 — Chain vs Agent: side-by-side

| Dimension | One-shot chain (LCEL) | SQL Agent |
|-----------|----------------------|-----------|
| LLM calls | 2 | 3–5 |
| Typical latency | 1–2 s | 3–6 s |
| Self-correction on bad SQL | ❌ crashes | ✅ retries |
| Schema discovery | Manual (you inject it) | Automatic |
| Multi-table joins | Sometimes fails | Handles reliably |
| Visibility into SQL | Always visible | Use `verbose=True` |
| Best for | Simple single-table queries | Production, unknown schema, complex joins |

Run the cell below to see both approaches answer the same question and compare latency.

In [ ]:
import time

question = "Which customers placed orders in the last 30 days?"

# ── One-shot chain ────────────────────────────────────────────────────────────
t0 = time.time()
r_chain = run_chain(question)
t_chain = time.time() - t0

# ── Agent (verbose off for clean output) ─────────────────────────────────────
agent_quiet = create_sql_agent(
    llm=llm, db=db,
    agent_type="tool-calling", top_k=20, verbose=False,
    agent_executor_kwargs={"handle_parsing_errors": True},
)
t0 = time.time()
r_agent = agent_quiet.invoke({"input": question})
t_agent = time.time() - t0

print(f"One-shot chain  ({t_chain:.1f}s)")
print(f"  SQL    : {r_chain['sql']}")
print(f"  Answer : {r_chain['answer'][:200]}")
print()
print(f"SQL Agent  ({t_agent:.1f}s)")
print(f"  Answer : {r_agent['output'][:200]}")

## Section 7 — Prompt Engineering

The LLM's behaviour is entirely determined by the text it receives. This section shows how to control it precisely.

### What LangChain injects by default

When you call `create_sql_agent`, LangChain builds a system message from two templates:

- **`SQL_PREFIX`** — core instructions: use the SQL tools, only query existing columns, add a LIMIT, generate `{dialect}`-specific SQL. Contains `{dialect}` and `{top_k}` placeholders that LangChain fills at runtime.
- **`SQL_SUFFIX`** — a scratchpad placeholder for the ReAct loop (less relevant for `tool-calling`).

The `prefix=` parameter **replaces** `SQL_PREFIX`. The pattern used in production is:

```python
AGENT_PREFIX = YOUR_BUSINESS_RULES + SQL_PREFIX
```

Putting your rules *before* `SQL_PREFIX` gives them higher priority — the LLM reads from top to bottom.

In [ ]:
from langchain_community.agent_toolkits.sql.prompt import SQL_PREFIX, SQL_SUFFIX

# Read the default prompts LangChain uses — these are the instructions every
# SQL agent receives before you add any of your own business rules.
print("── SQL_PREFIX (default agent system prompt) ─────────────────────")
print(SQL_PREFIX)

print("\n── SQL_SUFFIX ───────────────────────────────────────────────────")
print(SQL_SUFFIX)

print("\n── Placeholders filled at runtime ───────────────────────────────")
print(f"  {{dialect}} → '{db.dialect}' (from SQLDatabase.from_uri)")
print(f"  {{top_k}}   → 20 (from the top_k= argument)")

### 7a — Adding business rules (`prefix=`)

Business rules are domain facts the LLM cannot infer from the schema alone.
Examples for a retail database:
- Don't count cancelled orders as sales
- "last month" means 30 days, not a calendar month
- Revenue = quantity × unit_price (not product.price)

We write these as a plain English string and prepend it to `SQL_PREFIX`.

### 7b — Writing effective rules

Good rules are:
- **Specific** — say exactly what column to filter and what value to exclude
- **Absolute** — "NEVER" beats "try to avoid"
- **Contextual** — define ambiguous terms ("revenue", "last month", "sold")

Below we test weak vs strong phrasing on the same question.

In [ ]:
# ── Business rules used in production ────────────────────────────────────────
_BUSINESS_RULES = """You are a helpful data analyst assistant for a retail company.

Business rules — apply these to EVERY query without exception:
- NEVER include orders with status = 'cancelled' when calculating sales, quantities, or revenue.
- "sold", "top products", "best sellers", "revenue" always mean non-cancelled orders only.
  Always add the filter: WHERE orders.status != 'cancelled'
- Valid order statuses are: pending, shipped, delivered, cancelled.
- "last month" means the past 30 days (use datetime('now', '-30 days') for SQLite).
- "yesterday" means the previous calendar day (use date('now', '-1 day')).
- "this week" means the past 7 days (use datetime('now', '-7 days')).
- Sales volume: SUM(order_items.quantity) grouped by product.
- Revenue: SUM(order_items.quantity * order_items.unit_price).

"""

# Pattern: business rules first, then LangChain's standard SQL instructions.
# Rules before SQL_PREFIX = higher priority in the LLM's context window.
AGENT_PREFIX = _BUSINESS_RULES + SQL_PREFIX

# Helper — build an agent with any rule string
def make_agent(rules: str, verbose: bool = False) -> object:
    return create_sql_agent(
        llm=llm, db=db,
        agent_type="tool-calling",
        top_k=20,
        verbose=verbose,
        prefix=rules + SQL_PREFIX,
        agent_executor_kwargs={"handle_parsing_errors": True},
    )

# ── Weak vs strong phrasing ───────────────────────────────────────────────────
WEAK_RULES   = "\nYou are a data analyst. Try to exclude cancelled orders when possible.\n\n"
STRONG_RULES = (
    "\nYou are a data analyst.\n"
    "CRITICAL: NEVER include orders with status = 'cancelled' in ANY sales metric. "
    "This is a legal compliance requirement. No exceptions. "
    "Always add: WHERE orders.status != 'cancelled'\n\n"
)

q = "What is total revenue?"

r_weak   = make_agent(WEAK_RULES).invoke({"input": q})
r_strong = make_agent(STRONG_RULES).invoke({"input": q})

# Compute the correct answer locally
conn = sqlite3.connect(DB_PATH)
correct_revenue = conn.execute("""
    SELECT ROUND(SUM(oi.quantity * oi.unit_price), 2)
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.id
    WHERE o.status != 'cancelled'
""").fetchone()[0]
conn.close()

print(f"Weak rules   : {r_weak['output'][:200]}")
print(f"Strong rules : {r_strong['output'][:200]}")
print(f"\nCorrect revenue (cancelled excluded): ${correct_revenue}")

In [ ]:
# Build the production agent — uses the full business rules
agent_with_rules = make_agent(_BUSINESS_RULES, verbose=True)

# The acid test: does it exclude Ballpoint Pens (Frank's cancelled order)?
result = agent_with_rules.invoke({"input": "What is the top selling product by quantity?"})

print("\n── Answer (Ballpoint Pens x10 must NOT appear) ──────────────────")
print(result["output"])

### 7c — Few-shot examples

For tricky cases (date arithmetic, multi-join patterns), providing worked SQL examples directly in the prompt is more reliable than descriptions alone. The LLM pattern-matches on them.

In [ ]:
FEW_SHOT_RULES = _BUSINESS_RULES + """
Examples of correct SQL for common question patterns:

Q: Orders placed yesterday?
SQL: SELECT * FROM orders
     WHERE date(ordered_at) = date('now', '-1 day')
       AND status != 'cancelled'

Q: Revenue in the last 30 days?
SQL: SELECT ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
     FROM order_items oi
     JOIN orders o ON oi.order_id = o.id
     WHERE o.ordered_at >= datetime('now', '-30 days')
       AND o.status != 'cancelled'

Q: Top 5 products by units sold this week?
SQL: SELECT p.name, SUM(oi.quantity) AS units_sold
     FROM order_items oi
     JOIN orders o ON oi.order_id = o.id
     JOIN products p ON oi.product_id = p.id
     WHERE o.ordered_at >= datetime('now', '-7 days')
       AND o.status != 'cancelled'
     GROUP BY p.name
     ORDER BY units_sold DESC
     LIMIT 5

"""

agent_fewshot = make_agent(FEW_SHOT_RULES, verbose=True)
result = agent_fewshot.invoke({"input": "How much revenue did we make yesterday?"})

print("\n── Answer ───────────────────────────────────────────────────────")
print(result["output"])
# The LLM should use date('now', '-1 day') because of the few-shot example

## Section 8 — Memory and conversation history

### The problem

The SQL agent is **completely stateless**. Each `agent.invoke()` call starts with a blank slate — it has no memory of previous questions or answers. This means follow-up questions fail:

```
User: "What were the top 3 products last month?"
Agent: "Wireless Mouse, Mechanical Keyboard, USB-C Hub"

User: "Which customers bought them?"          ← "them" has no referent
Agent: "I'm not sure what 'them' refers to."  ← correct but useless
```

### The solution

Maintain a list of past Q&A turns in your own code, and **prepend** the recent history as plain text to every new question before sending it to the agent. The agent receives enough context to resolve references.

```
Previous conversation:
  User: What were the top 3 products last month?
  Assistant: Wireless Mouse, Mechanical Keyboard, USB-C Hub

Current question: Which customers bought them?
```

### Two implementations

We'll build both and compare:
1. **Manual plain-text history** — what the production chatbot uses. Simple, transparent, easy to debug.
2. **`InMemoryChatMessageHistory` + `trim_messages`** — the LangChain 1.x way. Stores typed message objects; useful when you need to integrate with other LangChain components.

### 8a — Manual plain-text history

Store the last N Q&A turns as a list of dicts `{"question": ..., "answer": ...}`.
Before each call, serialise the tail of the list as plain text and prepend it to the question.

**Why store the original question, not the enriched prompt?**
If you store the enriched prompt (which already contains previous history), the history grows exponentially — history-of-history-of-history. Store only the raw question; the enrichment is rebuilt fresh each turn.

In [ ]:
HISTORY_WINDOW = 5   # number of past turns to include in each prompt

def build_prompt_with_history(question: str, history: list[dict]) -> str:
    """Prepend the last HISTORY_WINDOW turns to the current question.

    Returns the question unchanged if history is empty.
    """
    if not history:
        return question
    lines = ["Previous conversation:"]
    for turn in history:
        lines.append(f"  User: {turn['question']}")
        lines.append(f"  Assistant: {turn['answer']}")
    lines.append(f"\nCurrent question: {question}")
    return "\n".join(lines)


history: list[dict] = []   # session state — a list of {question, answer} dicts

def ask(question: str) -> str:
    """One turn of conversation with history context.

    Requires: agent_with_rules defined in Section 7 (cell above this section).
    """
    # Build the enriched prompt from the last HISTORY_WINDOW turns
    prompt = build_prompt_with_history(question, history[-HISTORY_WINDOW:])
    print(f"── Prompt sent to agent ─────────────────────────────────────")
    print(prompt)
    print(f"─────────────────────────────────────────────────────────────\n")

    result = agent_with_rules.invoke({"input": prompt})
    answer = result["output"]

    # Store the ORIGINAL question (not the enriched prompt) to avoid history explosion
    history.append({"question": question, "answer": answer})
    return answer

In [ ]:
# Turn 1 — fresh session, no history yet; prompt = question unchanged
print("Answer 1:", ask("What were the top 3 products sold last month?"))

In [ ]:
# Turn 2 — "them" refers to the products named in turn 1.
# The agent resolves this because turn 1 is now in the prompt history.
print("Answer 2:", ask("Which customers bought them?"))

In [ ]:
# Turn 3 — "this week instead of last month" references both turns 1 and 2.
print("Answer 3:", ask("What about this week instead of last month?"))

In [ ]:
# Inspect the history list after 3 turns
print("Session history:")
for i, turn in enumerate(history, 1):
    print(f"\nTurn {i}")
    print(f"  Q: {turn['question']}")
    print(f"  A: {turn['answer'][:120]}...")

### 8b — The window size trade-off

The history window controls how much context the agent sees:

- **Too small (window = 1):** Follow-up questions that reference turns older than 1 step back will lose context — the agent can't resolve "them" if "them" was defined 2 turns ago.
- **Too large (window = 20):** Every prompt becomes huge, costing more tokens and increasing latency.
- **Production default:** 5 turns is a good balance for most NLQ use cases.

Run the cell below to see what the prompt looks like with a narrow window vs a full window.

In [ ]:
# Demonstrate the window size trade-off using the history built in 8a.
# Run cells 35-37 first so the history list has 3 turns.

sep = chr(0x2500) * 50

print(f'{sep} window=1 (only most recent turn)')
print(build_prompt_with_history('What about this week?', history[-1:]))

print()
print(f'{sep} window=3 (all 3 turns — full context)')
print(build_prompt_with_history('What about this week?', history[-3:]))

# With window=1 the agent sees only the last Q&A pair.
# Cross-turn references ('them' from turn 1) are invisible.
# With window=3 the full context is preserved.

### 8c — `InMemoryChatMessageHistory` (LangChain 1.x)

LangChain 1.x provides `InMemoryChatMessageHistory` from `langchain_core.chat_history`.
It stores messages as typed objects (`HumanMessage` / `AIMessage`) rather than plain dicts.

`get_buffer_string()` serialises them to the same plain text format we produce manually.
The two approaches are functionally equivalent; `InMemoryChatMessageHistory` is more
useful when you need to pass history into other LangChain components.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory, get_buffer_string

# Store messages as typed objects
lc_history = InMemoryChatMessageHistory()
lc_history.add_user_message("What were the top 3 products last month?")
lc_history.add_ai_message("Wireless Mouse, Mechanical Keyboard, USB-C Hub")
lc_history.add_user_message("Which customers bought them?")
lc_history.add_ai_message("Alice, Bob, Carol, Eva, Grace")

print("── Stored messages (typed objects) ─────────────────────────────")
for msg in lc_history.messages:
    print(f"  {type(msg).__name__:<14}: {msg.content}")

print("\n── Serialised to plain text via get_buffer_string() ─────────────")
print(get_buffer_string(lc_history.messages))

In [ ]:
# Side-by-side: the two approaches produce equivalent prompts
manual = build_prompt_with_history(
    "What region are they from?",
    [{"question": "What were the top 3 products last month?", "answer": "Wireless Mouse, Mechanical Keyboard, USB-C Hub"},
     {"question": "Which customers bought them?",             "answer": "Alice, Bob, Carol, Eva, Grace"}]
)

lc_text = get_buffer_string(lc_history.messages)

print("── Manual plain-text approach ───────────────────────────────────")
print(manual)

print("\n── LangChain InMemoryChatMessageHistory ─────────────────────────")
print(lc_text)
print("Current question: What region are they from?")
print()
# Both produce "Human: ..." / "AI: ..." or "User: ..." / "Assistant: ..."
# which the LLM treats identically as prior context.

### 8d — Windowing with `trim_messages`

`InMemoryChatMessageHistory` grows forever. To enforce a window, use `trim_messages` from `langchain_core.messages`.

**`token_counter` options:**
- `token_counter=len` — counts *messages* (1 message = 1 "token"). Simple, fast, no LLM call.
- `token_counter=lambda msgs: sum(len(m.content) for m in msgs)` — counts *characters*. Useful for longer answers.
- `token_counter=llm` — uses the model's actual tokeniser. Most accurate but makes an API call.

**`start_on="human"`** ensures the window always starts at the beginning of a turn (never cuts a pair in half).

In [ ]:
from langchain_core.messages import trim_messages, HumanMessage, AIMessage

# Build a history with 5 turns to experiment on
full_hist = InMemoryChatMessageHistory()
turns = [
    ("What were the top 3 products last month?", "Wireless Mouse, Mechanical Keyboard, USB-C Hub"),
    ("Which customers bought them?",             "Alice, Bob, Carol, Eva, Grace"),
    ("What region are they from?",               "North, South, East, West"),
    ("Which region had the most orders?",        "North had 3 orders"),
    ("What was the total revenue?",              "Revenue was $847.82 excluding cancelled orders"),
]
for q, a in turns:
    full_hist.add_user_message(q)
    full_hist.add_ai_message(a)

print(f"Total messages: {len(full_hist.messages)}  ({len(full_hist.messages)//2} turns)")

# ── Window by message count: keep last 3 turns = 6 messages ──────────────────
windowed_count = trim_messages(
    full_hist.messages,
    max_tokens=6,          # max 6 messages to keep
    token_counter=len,     # len(messages) — counting messages, not actual tokens
    strategy="last",       # keep the most recent
    include_system=False,
    allow_partial=False,
    start_on="human",      # always begin at the start of a turn pair
)
print(f"\nWindow by count (last 3 turns = 6 messages):")
for msg in windowed_count:
    print(f"  {type(msg).__name__:<14}: {msg.content[:60]}")

# ── Window by character budget: keep turns that fit within 150 chars ──────────
windowed_chars = trim_messages(
    full_hist.messages,
    max_tokens=150,
    token_counter=lambda msgs: sum(len(m.content) for m in msgs),
    strategy="last",
    include_system=False,
    allow_partial=False,
    start_on="human",
)
print(f"\nWindow by character budget (150 chars):")
for msg in windowed_chars:
    print(f"  {type(msg).__name__:<14}: {msg.content[:60]}")

In [ ]:
# build_prompt_from_lc_history: the LangChain-native equivalent of build_prompt_with_history().
# Combines InMemoryChatMessageHistory + trim_messages into a plain-text prompt string.
def build_prompt_from_lc_history(
    question: str,
    hist: InMemoryChatMessageHistory,
    window: int = 5,
) -> str:
    """Serialise the last `window` turns from an InMemoryChatMessageHistory into a prompt."""
    trimmed = trim_messages(
        hist.messages,
        max_tokens=window * 2,    # window turns * 2 messages per turn
        token_counter=len,
        strategy="last",
        include_system=False,
        allow_partial=False,
        start_on="human",
    )
    if not trimmed:
        return question
    lines = ["Previous conversation:"]
    for msg in trimmed:
        role = "User" if isinstance(msg, HumanMessage) else "Assistant"
        lines.append("  " + role + ": " + msg.content)
    lines.append("")
    lines.append("Current question: " + question)
    return "\n".join(lines)

sep = chr(0x2500) * 50

print(f'{sep} window=2 (last 2 turns)')
print(build_prompt_from_lc_history("What about this week?", full_hist, window=2))

print()
print(f'{sep} window=1 (only last turn -- earlier references are lost)')
print(build_prompt_from_lc_history("What about this week?", full_hist, window=1))


## Section 9 — Agent internals

What does the agent actually send to the LLM? Inspecting the prompt template
and the tool definitions reveals how the tool-calling loop works under the hood.


In [ ]:
# Inspect the prompt template the agent uses.
# agent.prompt.messages is the internal attribute in LangChain 1.x.
# Wrapped in try/except — if the attribute path doesn't exist in your build,
# a helpful fallback is printed instead of crashing the notebook.
print('-- Agent prompt messages --')
try:
    prompt_messages = agent_with_rules.agent.prompt.messages
except AttributeError:
    prompt_messages = []
    print('Note: agent.prompt not accessible in this LangChain build.')
    print('The system prompt is AGENT_PREFIX (Section 7) + SQL_PREFIX.')
    print('Inspect it with:  print(AGENT_PREFIX)')

for msg in prompt_messages:
    msg_type = type(msg).__name__
    if hasattr(msg, 'content') and msg.content:
        print(f'[{msg_type}] {str(msg.content)[:600]}')
    elif hasattr(msg, 'prompt'):
        print(f'[{msg_type}] template: {str(msg.prompt.template)[:600]}')
    else:
        print(f'[{msg_type}] (dynamic placeholder)')
    print()

In [ ]:
# Inspect the four tools the LLM can call.
# These are presented to the LLM as function definitions in the API request.
print("── Agent tools ──────────────────────────────────────────────────")
for tool in agent_with_rules.tools:
    print(f"Name : {tool.name}")
    print(f"Desc : {tool.description[:250]}")
    print()

# The LLM decides which tool to call based on these descriptions.
# e.g. if it doesn't know the table names, it calls sql_db_list_tables first.

The agent's tool descriptions are what guide the LLM's decisions. When the LLM is uncertain about table structure it calls `sql_db_schema`; when it has a query ready it calls `sql_db_query`. It can call `sql_db_query_checker` to self-review SQL before execution.

## Section 10 — Mini chatbot

Everything so far assembled into one ~40-line function that mirrors the production `app.py` exactly — same guard, same history, same agent. No Chainlit needed.

**Components assembled:**
1. SQL safety guard — blocks destructive statements if the user sends raw SQL
2. History builder — prepends last 5 turns to the prompt
3. Production agent — `tool-calling`, business rules, `top_k=20`
4. Session state — a plain list accumulating Q&A turns

In [ ]:
import re

# ── 1. SQL Safety Guard ───────────────────────────────────────────────────────
# Only runs when the user's input looks like raw SQL (first word is a SQL keyword).
# Plain-English questions ("What were the top products?") pass straight through.

_DESTRUCTIVE = re.compile(
    r"^\s*(DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|CREATE|REPLACE|MERGE)\b",
    re.IGNORECASE,
)
_CTE_PREFIX = re.compile(r"^\s*WITH\b.*?AS\s*\(", re.IGNORECASE | re.DOTALL)
_SQL_FIRST_WORDS = {"SELECT","WITH","DROP","DELETE","UPDATE","INSERT","ALTER","TRUNCATE","CREATE"}

def is_safe(sql: str) -> bool:
    """Return True if the input is a read-only SELECT (or CTE→SELECT), False otherwise."""
    clean = re.sub(r"--.*$",    "", sql, flags=re.MULTILINE)  # strip line comments
    clean = re.sub(r"/\*.*?\*/","", clean, flags=re.DOTALL)   # strip block comments
    if _DESTRUCTIVE.match(clean):
        return False
    body = _CTE_PREFIX.sub("", clean).strip()                 # strip leading WITH clause
    return body.upper().startswith("SELECT")

def looks_like_sql(text: str) -> bool:
    """Cheap pre-filter: True if first word is a SQL keyword."""
    tokens = text.split()
    return bool(tokens) and tokens[0].upper() in _SQL_FIRST_WORDS

# ── 2. Production agent (verbose=False for clean REPL output) ─────────────────
final_agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="tool-calling",
    top_k=20,
    verbose=False,    # flip to True to see every tool call
    prefix=AGENT_PREFIX,
    agent_executor_kwargs={"handle_parsing_errors": True},
)

# ── 3. Session state ──────────────────────────────────────────────────────────
session_history: list[dict] = []

# ── 4. Chat function ──────────────────────────────────────────────────────────
def chat(question: str) -> str:
    """Process one user turn: guard → history → agent → update history → return answer."""
    question = question.strip()

    # Guard: only check if input looks like raw SQL
    if looks_like_sql(question) and not is_safe(question):
        return "Sorry, I can only run read-only queries against the database."

    # Prepend recent history so the agent can resolve follow-up references
    prompt = build_prompt_with_history(question, session_history[-HISTORY_WINDOW:])

    result = final_agent.invoke({"input": prompt})
    answer = result.get("output", "I couldn't find an answer to that.")

    # Store original question (not the enriched prompt) to prevent history explosion
    session_history.append({"question": question, "answer": answer})
    return answer

print("Chatbot ready. Call  chat('your question')  to interact.")

In [ ]:
print(chat("What were the top 3 products by units sold?"))

In [ ]:
# Follow-up — "the top product" refers to turn 1's answer
print(chat("Which customers bought the top product?"))

In [ ]:
print(chat("How many orders are still pending?"))

In [ ]:
# Safety guard test — destructive SQL is blocked before it reaches the agent
print(chat("DROP TABLE orders"))
print(chat("DELETE FROM customers WHERE 1=1"))

# Safe SELECT passes through
print(chat("SELECT COUNT(*) FROM orders"))

## Section 11 — Exercises

Work through these to solidify each concept. Each one references the section where the relevant code lives.

---

**Prompt engineering (Section 7)**

1. Add a new business rule: `"'active customer' means a customer who has placed an order within the last 90 days"`. Ask the agent `"How many active customers do we have?"` and verify the SQL uses `datetime('now', '-90 days')`.
2. Change the few-shot examples in Section 7c to use wrong SQL on purpose (e.g. omit the cancelled filter). Does it hurt the answer?
3. Remove `_BUSINESS_RULES` entirely (pass only `SQL_PREFIX` as `prefix=`). Run the cancelled-order test again. Does the answer change?

---

**Memory (Section 8)**

4. Change `HISTORY_WINDOW = 1`. Ask a 3-turn conversation where turn 3 refers to something said in turn 1. What breaks?
5. Change `token_counter=len` to `token_counter=lambda msgs: sum(len(m.content) for m in msgs)` in `build_prompt_from_lc_history`. How does the retained window change when answers are long vs short?
6. `InMemoryChatMessageHistory` grows forever. Implement a helper that caps it at 20 messages: call `hist.clear()` and re-add only the last 20.

---

**Chain vs Agent (Sections 4–6)**

7. Ask `run_chain("Which products were bought by customers in the North region who placed an order in the last 30 days?")` — a 3-table join. Does it succeed? Try the same with `agent.invoke()`. Which handles it better?
8. Increase `verbose=True` on the agent and run a complex question. Count how many tool calls it makes. Is there a pattern?
9. Make the chain crash deliberately: call `run_chain("Show me all rows from order_totals")` (table doesn't exist). What error do you get? What does the agent do with the same question?

---

**Schema extension**

10. Add a `reviews` table:
    ```python
    cur.execute("CREATE TABLE reviews (id INTEGER PRIMARY KEY, order_id INTEGER REFERENCES orders(id), rating INTEGER, comment TEXT)")
    cur.execute("INSERT INTO reviews VALUES (1,1,5,'Great!'),(2,2,3,'OK'),(3,5,4,'Good')")
    conn.commit()
    ```
    Reconnect: `db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")`. Ask the agent: `"What is the average rating for Electronics products?"` — it now needs a 4-table join.

11. Try `db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}", include_tables=["orders", "order_items"])`. The agent can only see 2 tables. What questions now fail?

---

**Error handling**

12. Ask `chat("Show me the data")`. How does the agent respond when the question is completely ambiguous?
13. Ask `chat("What are the subscription tiers?")`. What does the agent say about a concept that doesn't exist in the schema?

## Section 12 — Concept summary

```
┌─────────────────────────────────────────────────────────────────────┐
│ COMPONENT              WHAT IT IS / DOES                            │
├─────────────────────────────────────────────────────────────────────┤
│ SQLDatabase            Python wrapper around SQLAlchemy.            │
│                        Provides schema introspection + db.run(sql). │
│                        The LLM never touches the DB directly.       │
│                                                                     │
│ One-shot chain (LCEL)  question → [LLM 1] SQL → db.run →           │
│                                  [LLM 2] format answer              │
│                        2 LLM calls. Fast. No retry on bad SQL.      │
│                        Good for: simple queries, tight latency.     │
│                                                                     │
│ SQL Agent              question → loop {pick tool → call → observe} │
│                                  → answer                           │
│                        3-5 LLM calls. Self-correcting.              │
│                        Good for: multi-table joins, production.     │
│                                                                     │
│ agent_type             "tool-calling" = JSON calls (use this)       │
│                        "zero-shot-react" = text parsing (fragile)   │
│                                                                     │
│ prefix=                Replaces SQL_PREFIX in agent system prompt.  │
│                        Pattern: BUSINESS_RULES + SQL_PREFIX         │
│                        {dialect} and {top_k} filled by LangChain.  │
│                                                                     │
│ Business rules         Plain-English constraints injected via prefix.│
│                        Be specific and absolute. NEVER > try to.   │
│                        Few-shot SQL examples help date arithmetic.  │
│                                                                     │
│ Memory                 The agent is stateless — no built-in history.│
│                        Keep a turns list yourself, serialise to     │
│                        plain text, prepend to each new question.    │
│                        Window size = context richness vs token cost.│
│                                                                     │
│ InMemoryChatMsg        LangChain 1.x replacement for the removed    │
│ History                langchain.memory classes. Stores typed       │
│                        HumanMessage / AIMessage objects.            │
│                        Use trim_messages() for windowing.           │
│                                                                     │
│ trim_messages()        Trims a message list to fit a budget.        │
│                        token_counter=len  → count messages          │
│                        strategy="last"   → keep most recent turns   │
│                        start_on="human"  → never split a turn pair  │
└─────────────────────────────────────────────────────────────────────┘

Full request flow (production chatbot):

  user types question
    │
    ├─ looks_like_sql()? → yes → is_safe()? → no → block (read-only error)
    │                                        ↓ yes
    │                               agent.invoke()
    │ no
    ↓
  build_prompt_with_history(question, last 5 turns)
    ↓
  agent.invoke(enriched_prompt)
    ├── sql_db_list_tables
    ├── sql_db_schema(relevant tables)
    ├── sql_db_query(generated SQL)
    └── (retry with corrected SQL if error)
    ↓
  plain-English answer
    ↓
  history.append({question, answer})
    ↓
  return answer to user
```